# 实验二：截图证据补充

本Notebook仅补充真实源码、原在线响应归档、CSV与只读MySQL核对证据。它不重新采集网页、不修改数据库、不覆盖主Notebook或正式CSV。红框是Notebook输出中的源码展示，并不冒充IDE、Chrome Elements或MySQL终端界面。密码从临时环境变量或交互输入读取，不写入文档。

In [1]:
from pathlib import Path
import inspect, html, json, hashlib, os
from datetime import datetime
from getpass import getpass
from IPython.display import HTML, display
from bs4 import BeautifulSoup
import pymysql
import collect_houseinfo as collector
import fetch_firecrawl as online
import import_houseinfo as importer

LAB = Path.cwd()
EVIDENCE = LAB.parent / "work" / "supplement_evidence"
EVIDENCE.mkdir(parents=True, exist_ok=True)
FIELDS = collector.FIELDS
manifest = json.loads((LAB / "data/collection_summary.json").read_text(encoding="utf-8"))
css = """body{margin:0;background:#f3f5f7;color:#18212d;font-family:Arial,'Songti SC','SimSun',sans-serif;padding:24px}main{background:white;padding:24px;border:1px solid #d5dce5}h1{font-size:26px;margin:0 0 8px}h2{font-size:20px;margin:14px 0 8px}p{font-size:16px;line-height:1.55}pre{font:16px/1.45 Consolas,Menlo,monospace;white-space:pre-wrap;overflow-wrap:anywhere;margin:8px 0}.source{border:2px solid #ce3e3e;padding:10px;background:#fafafa}.label{font-size:15px;color:#526071}.result{border-left:4px solid #238636;padding:10px;background:#f4fff7}.columns{display:grid;grid-template-columns:1fr 1fr;gap:20px;align-items:start}table{border-collapse:collapse;width:100%;font-size:16px}th,td{border:1px solid #c8d0d9;padding:8px;text-align:left;overflow-wrap:anywhere}th{background:#edf2f7}.key td{background:#fff3e8;border-top:2px solid #cc4c39;border-bottom:2px solid #cc4c39}.note{padding:9px;background:#fff8dd}code{font-family:Consolas,Menlo,monospace}a{color:#0969da}"""

def source_block(function, selected=None):
    lines, first = inspect.getsourcelines(function)
    numbered = []
    last_added = None
    for offset, line in enumerate(lines):
        number = first + offset
        if selected is None or any(low <= number <= high for low, high in selected):
            if last_added is not None and number > last_added + 1:
                numbered.append("     …（以下省略的行仍保留在原文件中）")
            numbered.append(f"{number:>3}  {line.rstrip()}")
            last_added = number
    name = Path(inspect.getsourcefile(function)).name
    label = f"{name} · {function.__name__}() · 实际源码行号"
    return f'<div class="label">{html.escape(label)}</div><pre class="source">{html.escape(chr(10).join(numbered))}</pre>'

def result_block(text):
    return '<pre class="result">' + html.escape(str(text)) + '</pre>'

def save_page(filename, title, content, note):
    stamp = datetime.now().astimezone().isoformat(timespec="seconds")
    page = f'<!doctype html><html lang="zh"><meta charset="utf-8"><title>{html.escape(title)}</title><style>{css}</style><body><main><h1>{html.escape(title)}</h1><p class="label">截图证据补充.ipynb · 本单元真实执行输出 · {stamp}</p><p class="note">{html.escape(note)}</p>{content}</main></body></html>'
    (EVIDENCE / filename).write_text(page, encoding="utf-8")
    display(HTML(page))

print("补充证据环境已就绪：只读取在线归档、正式CSV和MySQL；不重新联网采集、不修改数据库。")


补充证据环境已就绪：只读取在线归档、正式CSV和MySQL；不重新联网采集、不修改数据库。


In [2]:
first_page = manifest["pages"][0]
first_html = (LAB / "html/online" / first_page["file"]).read_text(encoding="utf-8")
assert hashlib.sha256(first_html.encode()).hexdigest() == first_page["html_sha256"]
soup = BeautifulSoup(first_html, "html.parser")
rows = soup.select("div.property")
first_record = collector.parse_house(rows[0])
output = [f"实际归档：html/online/{first_page['file']}；房源父容器：{len(rows)} 个"]
output.extend(f"{field}: {value}" for field, value in zip(FIELDS, first_record))

# 只修改内存副本，实际测试年代缺失分支；不改归档和正式记录。
copied_row = BeautifulSoup(str(rows[0]), "html.parser").select_one("div.property")
year_nodes = [node for node in copied_row.select(".property-content-info > p.property-content-info-text") if "年建造" in node.get_text()]
for node in year_nodes:
    node.decompose()
assert collector.parse_house(copied_row)[6] == ""
output.append("内存副本删除年代节点后：maketime == ''，缺失处理实际通过。")

area_node = next(node for node in rows[0].select(".property-content-info > p.property-content-info-text") if node.get_text(strip=True) == first_record[4])
price_node = rows[0].select_one(".property-price-average")
original_nodes = str(area_node) + "\n" + str(price_node)
left = source_block(collector.text_of) + source_block(collector.parse_house)
right = source_block(collector.parse_page, [(45, 46), (50, 53)]) + result_block("\n".join(output)) + '<h2>首条面积与单价的归档HTML原文</h2>' + '<pre class="source">' + html.escape(original_nodes) + '</pre>'
save_page("01_parser.html", "八字段解析：实际源码与首条输出", '<div class="columns"><section>'+left+'</section><section>'+right+'</section></div>', "代码来自现有源文件，保留真实行号；本次解析已保存的真实在线HTML。这是Notebook输出中的源码展示，不是IDE或Chrome Elements截图。")


In [3]:
# 读取并校验原在线请求清单；此单元不发起新的网络请求。
log_lines = ["原在线请求日志（读取 data/collection_summary.json；本单元没有重新在线采集）："]
log_lines.append("采集开始：" + manifest["started_at"])
for entry in manifest["pages"]:
    archived = (LAB / "html/online" / entry["file"]).read_text(encoding="utf-8")
    assert hashlib.sha256(archived.encode()).hexdigest() == entry["html_sha256"]
    actual_records, info = collector.parse_page(archived, entry["requested_url"])
    assert len(actual_records) == entry["count"]
    assert info["next_href"] == entry["next_href"] and info["last_page"] == entry["last_page"]
    if entry["page"] in (1, 2, 10):
        log_lines.append(f"第{entry['page']}页 | {entry['requested_at']} | {entry['count']}条")
        log_lines.append("  请求URL：" + entry["requested_url"])
        log_lines.append("  href：" + entry["next_href"])
        log_lines.append("  下一页：" + str(entry["next_url"]) + "；末页=" + str(entry["last_page"]))
assert manifest["complete"] is True and manifest["pages"][-1]["last_page"] is True
log_lines.extend(["原日志停止原因：" + manifest["stop_reason"], "原采集结束：" + manifest["finished_at"], "本次逐页核对归档SHA256、实际条数与分页状态：10页全部通过。"])

left = source_block(collector.parse_page, [(54, 75)])
right = source_block(online.collect, [(102, 108), (127, 136), (157, 168)])
content = '<div class="columns"><section><h2>提取href、urljoin及末页判断</h2>'+left+'</section><section><h2>真实采集循环源码（节选）</h2>'+right+'</section></div>' + result_block("\n".join(log_lines))
save_page("02_pagination.html", "分页源码与原在线请求日志核验", content, "红框展示原始程序真实源码；下方为此前在线请求日志的读取结果。本单元仅回放和核验归档，不能称为本次重新在线翻页。")


In [4]:
# 实际回读正式CSV；写入验证在内存中进行，不覆盖正式数据文件。
import csv as csv_module
import io
csv_records = importer.read_records(LAB / "data/houseinfo.csv")
stream = io.StringIO(newline="")
writer = csv_module.writer(stream)
writer.writerow(FIELDS)
writer.writerows(csv_records)
stream.seek(0)
readback = list(csv_module.DictReader(stream))
assert [tuple(row[field] for field in FIELDS) for row in readback] == csv_records
official_bytes = (LAB / "data/houseinfo.csv").read_bytes()
assert official_bytes.startswith(b"\xef\xbb\xbf")
facts = ["正式CSV：data/houseinfo.csv", "文件实际含UTF-8 BOM，与utf-8-sig一致。", "八列表头：" + ", ".join(FIELDS), f"正式CSV回读：{len(csv_records)}条；八字段内存写入后回读：{len(readback)}条，逐条一致。", "首条入库参数（按固定FIELDS顺序生成的8元素元组）："]
facts.extend(f"参数{i+1} {field} = {value}" for i, (field,value) in enumerate(zip(FIELDS,csv_records[0])))
content = '<div class="columns"><section>'+source_block(collector.save_csv)+'</section><section>'+source_block(importer.read_records)+'</section></div>' + result_block("\n".join(facts))
save_page("03_csv_parameters.html", "CSV编码、表头、回读与八列入库参数", content, "源码由inspect直接读取原文件；本单元实际回读正式CSV，并在内存验证写入/回读一致性，不覆盖正式CSV，不执行入库。")


In [5]:
# 通过只读事务查询数据库，不执行INSERT、UPDATE、DELETE或DDL。
password = os.environ.get("LAB02_MYSQL_PASSWORD") or getpass("MySQL密码（不保存）：")
connection = pymysql.connect(host="127.0.0.1", port=3306, user="student", password=password,
                             database="smartpark_lab", charset="utf8mb4", autocommit=False)
try:
    with connection.cursor() as cursor:
        cursor.execute("SET SESSION TRANSACTION READ ONLY")
        cursor.execute("START TRANSACTION READ ONLY")
        cursor.execute("SELECT title,house,address,`struct`,area,floor,maketime,price FROM houseinfo")
        database_records = list(cursor.fetchall())
finally:
    connection.rollback()
    connection.close()
del password
assert len(database_records) == len(csv_records) == 498
assert set(database_records) == set(csv_records)

all_archive_records = []
record_source = {}
for entry in manifest["pages"]:
    actual_html = (LAB / "html/online" / entry["file"]).read_text(encoding="utf-8")
    actual_records, _ = collector.parse_page(actual_html, entry["requested_url"])
    for position, record in enumerate(actual_records, 1):
        all_archive_records.append(record)
        record_source.setdefault(record, (entry["page"], position, entry["requested_url"]))
archive_unique = list(dict.fromkeys(all_archive_records))
assert archive_unique == csv_records
sections = [result_block(f"本次只读查询：数据库{len(database_records)}条；CSV{len(csv_records)}条；全部八字段集合一致。\n在线归档600条，八字段去重498条，与CSV逐条逐字段一致。")]
for index in (0, 60, len(csv_records)-1):
    csv_record = csv_records[index]
    archive_record = archive_unique[index]
    database_record = next(record for record in database_records if record == csv_record)
    page, position, url = record_source[csv_record]
    sections.append(f'<h2>样本{index+1} · 第{page}页第{position}条房源 · CSV数据行{index+1}</h2><p class="label">来源：{html.escape(url)}；对应html/online/page_{page:02}.html</p>')
    table = '<table><thead><tr><th>字段</th><th>在线归档HTML解析</th><th>CSV回读</th><th>MySQL只读查询</th><th>结果</th></tr></thead><tbody>'
    for column, field in enumerate(FIELDS):
        same = archive_record[column] == csv_record[column] == database_record[column]
        assert same
        cls = ' class="key"' if field in ("area", "maketime", "price") else ''
        values = [field, archive_record[column], csv_record[column], database_record[column], "一致"]
        table += f'<tr{cls}>' + ''.join('<td>'+html.escape(str(value))+'</td>' for value in values) + '</tr>'
    sections.append(table+'</tbody></table>')
save_page("04_sample_comparison.html", "首条、跨页与末条：HTML / CSV / MySQL八字段核对", ''.join(sections), "实际只读查询数据库并逐字段比较；橙色边框为面积、年代、单价。HTML列来自已校验的原在线响应归档，不代表本单元重新联网取得网页。")


字段,在线归档HTML解析,CSV回读,MySQL只读查询,结果
title,万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器,万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器,万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器,一致
house,万科金域学府翰林菁华里,万科金域学府翰林菁华里,万科金域学府翰林菁华里,一致
address,南岸学府大道二塘路59号,南岸学府大道二塘路59号,南岸学府大道二塘路59号,一致
struct,3室2厅1卫,3室2厅1卫,3室2厅1卫,一致
area,89.9㎡,89.9㎡,89.9㎡,一致
floor,低层(共27层),低层(共27层),低层(共27层),一致
maketime,2019年建造,2019年建造,2019年建造,一致
price,16465元/㎡,16465元/㎡,16465元/㎡,一致
字段,在线归档HTML解析,CSV回读,MySQL只读查询,结果
title,万科物业自营 朝中庭 业主诚心出售,万科物业自营 朝中庭 业主诚心出售,万科物业自营 朝中庭 业主诚心出售,一致
